In [1]:
from src import Transformer, train_model, map_target_column
import pandas as pd

In [2]:
df = pd.read_csv("dataset/bank-full_train_test.csv")
df = map_target_column(df, "y")
df_validation = pd.read_csv("dataset/bank-full_val.csv")
df_validation = map_target_column(df_validation, "y")

In [3]:
transformer = Transformer()
df = transformer.balance_dataset(df)
df = transformer.transform(df)
model, refence_df = train_model(df, target_column="y")

c:\Users\rober_aan06tc\Documents\EADA\Session - 5\class notes\nannyml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\rober_aan06tc\Documents\EADA\Session - 5\class notes\nannyml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
df_validation = transformer.transform(df_validation)
target = df_validation["y"]
df_validation = df_validation.drop("y", axis=1)

In [5]:
y = model.predict(df_validation)
y_proba = model.predict_proba(df_validation)

In [6]:
df_validation["prediction"] = y
df_validation["predicted_probability"] = y_proba[:,1]

In [7]:
df_validation.head()

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,previous,prediction,predicted_probability
0,40,0,580,1,0,16,5,192,1,-1,0,0,0.183874
1,47,0,3644,0,0,9,6,83,2,-1,0,0,0.339632
2,25,0,538,1,0,20,4,226,1,-1,0,0,0.221414
3,42,0,1773,0,0,9,4,311,1,336,1,1,0.850683
4,56,0,217,0,1,21,7,121,2,-1,0,0,0.176770


In [8]:
refence_df.head()

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,previous,prediction,predicted_probability,y
774,32,0,0,0,0,21,11,206,1,-1,0,0,0.488301,0
8337,52,0,195,1,0,18,2,220,1,63,5,0,0.372403,1
6213,46,0,7,0,0,17,6,110,2,-1,0,0,0.329767,0
6209,44,0,825,1,0,11,6,427,2,-1,0,0,0.393746,0
2362,37,0,1,1,0,6,5,608,1,-1,0,1,0.673017,1


In [9]:
import nannyml as nml

In [ ]:
# Using Confidence-based Performance Estimation for binary classification model performance estimation
chunk_size = 500
estimator = nml.CBPE(
    problem_type='classification_binary',
    y_pred_proba='predicted_probability',
    y_pred='prediction',
    y_true='y',
    metrics=['roc_auc'],
    chunk_size=chunk_size,
)

In [17]:
estimator = estimator.fit(refence_df)
estimated_performance = estimator.estimate(df_validation)

c:\Users\rober_aan06tc\Documents\EADA\Session - 5\class notes\nannyml\.venv\Lib\site-packages\nannyml\chunk.py:181: UserWarning:

The resulting number of chunks is too low. Please consider splitting your data in a different way or continue at your own risk.

c:\Users\rober_aan06tc\Documents\EADA\Session - 5\class notes\nannyml\.venv\Lib\site-packages\nannyml\chunk.py:181: UserWarning:

The resulting number of chunks is too low. Please consider splitting your data in a different way or continue at your own risk.



In [18]:
figure = estimated_performance.plot()
figure.show()